# McLaren Race Intelligence Platform — Data Preparation

## Overview

This notebook is the behind-the-scenes data engine for the **McLaren Race Intelligence Platform** Google Cloud marketing lab. It sources, cleans, and stages all the data assets the lab needs.

**What this notebook produces:**

| Asset | Destination | Used for |
|-------|-------------|----------|
| F1 race data (15 tables) | GCS `bq-staging/` (Parquet) | BigQuery load, BQML model training |
| FIA regulation PDFs | GCS `fia-docs/` | Gemini Enterprise RAG |
| McLaren driver + team profiles (JSON) | GCS `profiles/` | Gemini Enterprise structured data store |
| Profile embeddings (3072-dim) | GCS `profiles/` | Vector similarity search |

**Sections:**
1. Setup & Configuration
2. F1 Race Data — Source & Explore (Kaggle)
3. F1 Race Data — Clean & Stage to GCS (Parquet)
4. FIA Regulation Documents
5. McLaren Driver & Team Profiles
6. Verification & Data Manifest

**Requirements:**
- Colab Enterprise (IAM auth is automatic)
- Kaggle API credentials (see Section 1)
- Write access to `gs://class-demo`


---
## Section 0: Setup & Configuration

Install dependencies, auto-detect the GCP project, and define the constants used throughout the notebook. No manual project ID entry needed — Colab Enterprise provides credentials automatically.

In [ ]:
# Install required packages.
# kagglehub: clean interface for downloading Kaggle datasets
# google-genai: unified SDK for Gemini API via Vertex AI
# google-cloud-storage: GCS client for staging files
# pyarrow: Parquet serialization for GCS staging
!pip install -q kagglehub google-genai google-cloud-storage pandas tqdm pyarrow


In [ ]:
# Auto-detect the current GCP project from Colab Enterprise credentials.
# In Colab Enterprise, google.auth.default() returns the project associated
# with the notebook's runtime — no manual configuration needed.
import google.auth

credentials, PROJECT_ID = google.auth.default()

print(f"✅ Authenticated successfully")
print(f"   Project: {PROJECT_ID}")

# ── Constants ────────────────────────────────────────────────────────────────
# GCS staging locations
GCS_BUCKET       = "class-demo"
GCS_PREFIX       = "mclaren-f1"
GCS_BQ_STAGING   = f"{GCS_PREFIX}/bq-staging"   # Parquet files for BQ load
GCS_FIA_PREFIX   = f"{GCS_PREFIX}/fia-docs"
GCS_PROF_PREFIX  = f"{GCS_PREFIX}/profiles"

# Gemini model names
GEMINI_MODEL    = "gemini-2.5-flash"
EMBED_MODEL     = "gemini-embedding-001"
EMBED_DIM       = 3072   # maximum supported dimension for rich semantic capture

# Kaggle dataset slug
KAGGLE_DATASET  = "rohanrao/formula-1-world-championship-1950-2020"

# McLaren constructor reference ID in the Ergast dataset
MCLAREN_REF     = "mclaren"

print(f"\n📦 Staging bucket : gs://{GCS_BUCKET}/{GCS_PREFIX}/")
print(f"   BQ staging    : gs://{GCS_BUCKET}/{GCS_BQ_STAGING}/")
print(f"   FIA docs      : gs://{GCS_BUCKET}/{GCS_FIA_PREFIX}/")
print(f"   Profiles      : gs://{GCS_BUCKET}/{GCS_PROF_PREFIX}/")
print(f"🤖 Gemini model   : {GEMINI_MODEL}")
print(f"🔢 Embedding dims : {EMBED_DIM}")


In [ ]:
# Initialize the GCS client and confirm the staging bucket is accessible.
# BigQuery credentials are not needed here — the lab loads from GCS Parquet
# in a student-facing step, so there's no BQ setup in this notebook.
from google.cloud import storage

gcs_client = storage.Client(project=PROJECT_ID)

bucket = gcs_client.bucket(GCS_BUCKET)
if bucket.exists():
    print(f"✅ GCS bucket accessible: gs://{GCS_BUCKET}/")
else:
    print(f"❌ GCS bucket not found: gs://{GCS_BUCKET}/")
    print("   Verify bucket name and that your service account has Storage Object Admin role.")


In [10]:
# Kaggle credentials.
# Enter your Kaggle username and API key below.
# These are only used to download the F1 dataset in Section 1.
#
# To get your API key:
#   1. Go to https://www.kaggle.com/settings/account
#   2. Click "Create New Token" — this downloads kaggle.json
#   3. Your username and key are in that file
#
# If you're a student just reviewing this notebook, you can leave
# these blank — the rest of the notebook will work fine without them.

KAGGLE_USERNAME = "patrick.haggerty@roitraining.com"  # @param {type:"string"}
KAGGLE_KEY      = "KGAT_8255ff18b29f65ffb40039aef53f74cc"  # @param {type:"string"}

if KAGGLE_USERNAME and KAGGLE_KEY:
    import os
    os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
    os.environ["KAGGLE_KEY"]      = KAGGLE_KEY
    print("✅ Kaggle credentials set")
else:
    print("⚠️  Kaggle credentials not entered — skip to Section 2 if dataset already downloaded")

✅ Kaggle credentials set


---
## Section 1: F1 Race Data — Source & Explore

We use the **Ergast-derived Kaggle dataset** (rohanrao/formula-1-world-championship-1950-2020), which despite its name is updated through the 2024 season. It contains 14 tables covering every race, driver, constructor, result, qualifying session, pit stop, and lap time from 1950 to the present.

This section downloads the data and explores its structure so we understand exactly what we're working with before cleaning and staging it.


In [12]:
# Verify Kaggle credentials are available before attempting the download.
# Credentials were entered in the Section 0 configuration cell above.

import os

if not os.environ.get("KAGGLE_USERNAME") or not os.environ.get("KAGGLE_KEY"):
    print("❌ Kaggle credentials not found.")
    print("   Go back to the Section 0 configuration cell and enter your")
    print("   KAGGLE_USERNAME and KAGGLE_KEY, then re-run that cell.")
    raise SystemExit("Stopping — Kaggle credentials required to continue.")
else:
    print(f"✅ Kaggle credentials ready (user: {os.environ['KAGGLE_USERNAME']})")

✅ Kaggle credentials ready (user: patrick.haggerty@roitraining.com)


In [13]:
# Download the F1 dataset from Kaggle using kagglehub.
# kagglehub caches downloads, so re-running this cell is fast if you've
# already downloaded it in this session.
import kagglehub
import pathlib

print(f"Downloading dataset: {KAGGLE_DATASET}")
print("This may take a minute on first run...\n")

dataset_path = pathlib.Path(kagglehub.dataset_download(KAGGLE_DATASET))

csv_files = sorted(dataset_path.glob("*.csv"))

print(f"✅ Dataset downloaded to: {dataset_path}")
print(f"\n📁 CSV files found ({len(csv_files)} total):")
for f in csv_files:
    size_kb = f.stat().st_size / 1024
    print(f"   {f.name:<35} {size_kb:>8.1f} KB")

This may take a minute on first run...



100%|██████████| 6.28M/6.28M [00:00<00:00, 131MB/s]

Extracting files...


✅ Dataset downloaded to: /root/.cache/kagglehub/datasets/rohanrao/formula-1-world-championship-1950-2020/versions/24

📁 CSV files found (14 total):
   circuits.csv                             9.9 KB
   constructor_results.csv                214.2 KB
   constructor_standings.csv              309.8 KB
   constructors.csv                        17.1 KB
   driver_standings.csv                   863.1 KB
   drivers.csv                             92.2 KB
   lap_times.csv                        17209.4 KB
   pit_stops.csv                          433.3 KB
   qualifying.csv                         454.3 KB
   races.csv                              160.5 KB
   results.csv                           1681.6 KB
   seasons.csv                              4.5 KB
   sprint_results.csv                      24.2 KB
   status.csv                               2.1 KB


In [14]:
# Load all CSV tables into pandas DataFrames and display a schema summary.
# This gives us a clear picture of what's in each table before we clean and stage it.
import pandas as pd

TABLE_NAMES = [
    "circuits", "constructor_results", "constructor_standings",
    "constructors", "driver_standings", "drivers",
    "lap_times", "pit_stops", "qualifying",
    "races", "results", "seasons",
    "sprint_results", "status"
]

raw_tables = {}
missing = []

for name in TABLE_NAMES:
    path = dataset_path / f"{name}.csv"
    if path.exists():
        raw_tables[name] = pd.read_csv(path, low_memory=False)
    else:
        # Some versions of the dataset use slightly different filenames
        # e.g. constructorResults.csv vs constructor_results.csv
        camel = "".join(w.capitalize() for i, w in enumerate(name.split("_")))
        camel = camel[0].lower() + camel[1:]
        alt_path = dataset_path / f"{camel}.csv"
        if alt_path.exists():
            raw_tables[name] = pd.read_csv(alt_path, low_memory=False)
        else:
            missing.append(name)

print(f"✅ Loaded {len(raw_tables)} tables")
if missing:
    print(f"⚠️  Not found: {missing}")

print("\n── Table summary ──────────────────────────────────────────────────")
print(f"{'Table':<30} {'Rows':>8}  Columns")
print("-" * 65)
for name, df in sorted(raw_tables.items()):
    print(f"{name:<30} {len(df):>8,}  {list(df.columns)}")

✅ Loaded 14 tables

── Table summary ──────────────────────────────────────────────────
Table                              Rows  Columns
-----------------------------------------------------------------
circuits                             77  ['circuitId', 'circuitRef', 'name', 'location', 'country', 'lat', 'lng', 'alt', 'url']
constructor_results              12,625  ['constructorResultsId', 'raceId', 'constructorId', 'points', 'status']
constructor_standings            13,391  ['constructorStandingsId', 'raceId', 'constructorId', 'points', 'position', 'positionText', 'wins']
constructors                        212  ['constructorId', 'constructorRef', 'name', 'nationality', 'url']
driver_standings                 34,863  ['driverStandingsId', 'raceId', 'driverId', 'points', 'position', 'positionText', 'wins']
drivers                             861  ['driverId', 'driverRef', 'number', 'code', 'forename', 'surname', 'dob', 'nationality', 'url']
lap_times                       589,081 

In [15]:
# Quick McLaren-specific exploration to confirm the data covers what we need.
# We verify: McLaren exists as a constructor, find their first and last race,
# count total races and results, and list all drivers who raced for them.

constructors = raw_tables["constructors"]
results      = raw_tables["results"]
races        = raw_tables["races"]
drivers      = raw_tables["drivers"]

# Find McLaren's constructorId
mclaren_row = constructors[constructors["constructorRef"] == MCLAREN_REF]
if mclaren_row.empty:
    # Try lowercase match
    mclaren_row = constructors[constructors["constructorRef"].str.lower() == MCLAREN_REF]

assert not mclaren_row.empty, "McLaren not found in constructors table — check MCLAREN_REF constant"

MCLAREN_ID = mclaren_row.iloc[0]["constructorId"]
print(f"McLaren constructorId: {MCLAREN_ID}")
print(f"Full name: {mclaren_row.iloc[0]['name']}")
print(f"Nationality: {mclaren_row.iloc[0]['nationality']}")

# McLaren results
mclaren_results = results[results["constructorId"] == MCLAREN_ID].copy()
mclaren_races   = mclaren_results.merge(races[["raceId", "year", "name"]], on="raceId")

print(f"\n── McLaren race entries ───────────────────────────────────")
print(f"Total result rows  : {len(mclaren_results):,}")
print(f"Unique races       : {mclaren_results['raceId'].nunique():,}")
print(f"Year range         : {mclaren_races['year'].min()} – {mclaren_races['year'].max()}")

# All drivers who ever raced for McLaren
mclaren_driver_ids = mclaren_results["driverId"].unique()
mclaren_drivers_df = drivers[drivers["driverId"].isin(mclaren_driver_ids)].copy()
mclaren_drivers_df = mclaren_drivers_df.sort_values(["surname", "forename"])

print(f"\n── Unique McLaren drivers: {len(mclaren_drivers_df)} ─────────────────────")
for _, row in mclaren_drivers_df.iterrows():
    drv_races = mclaren_races[mclaren_races["driverId"] == row["driverId"]]
    years = sorted(drv_races["year"].unique())
    year_range = f"{min(years)}–{max(years)}" if len(years) > 1 else str(years[0])
    print(f"  {row['forename']} {row['surname']:<25} {year_range}")

McLaren constructorId: 1
Full name: McLaren
Nationality: British

── McLaren race entries ───────────────────────────────────
Total result rows  : 1,923
Unique races       : 929
Year range         : 1968 – 2024

── Unique McLaren drivers: 55 ─────────────────────
  Philippe Alliot                    1994
  Fernando Alonso                    2007–2018
  Michael Andretti                  1993
  Gerhard Berger                    1990–1992
  Mark Blundell                  1995
  Jo Bonnier                   1971
  Martin Brundle                   1994
  Jenson Button                    2010–2017
  Dave Charlton                  1974–1975
  David Coulthard                 1996–2004
  Mark Donohue                   1971
  Emerson Fittipaldi                1974–1975
  Peter Gethin                    1971
  Bruno Giacomelli                1977–1978
  Mike Hailwood                  1974
  Lewis Hamilton                  2007–2012
  David Hobbs                     1971–1974
  Denny Hulme        

## Section 1.5: Extend Dataset with 2025/2026 Data (Jolpica API)

Jolpica is the community-maintained successor to the deprecated Ergast API.
It uses identical endpoint structure and JSON schema, updated after each race weekend.

We fetch 2025 (full season) and 2026 (completed rounds only) and merge the
results into `raw_tables` so Section 2 stages everything to GCS in one pass.
No special handling needed — the merge cell normalises IDs and deduplicates.

**Tables updated:** `races`, `results`, `qualifying`, `driver_standings`,
`constructor_standings`, `sprint_results`, `drivers` (new entrants), `constructors` (new entrants), `status` (new status strings)


In [ ]:
import requests, time
import numpy as np

# ── Constants ─────────────────────────────────────────────────────────────────
JOLPICA_BASE   = "https://api.jolpi.ca/ergast/f1"
JOLPICA_YEARS  = [2025, 2026]
RATE_SLEEP     = 1.2   # seconds between successful calls
RATE_LIMIT_WAIT = 65   # seconds to wait after a 429 (1 min + buffer)

# ── Fetch helpers ─────────────────────────────────────────────────────────────
def jolpica_get(url, retries=5):
    """GET a Jolpica URL with retry/backoff and explicit 429 handling."""
    for attempt in range(retries):
        try:
            resp = requests.get(url, timeout=30)

            # Sprint/not-found endpoint — return None cleanly
            if resp.status_code == 404:
                time.sleep(RATE_SLEEP)
                return None

            # Rate-limited — honour Retry-After header or fall back to 65s
            if resp.status_code == 429:
                retry_after = int(resp.headers.get("Retry-After", RATE_LIMIT_WAIT))
                retry_after = max(retry_after, 30)   # never wait less than 30s
                print(f"  ⏳ 429 rate-limit — waiting {retry_after}s "
                      f"(attempt {attempt+1}/{retries}) for: {url.split('?')[0]}")
                time.sleep(retry_after)
                continue                              # retry without counting as error

            resp.raise_for_status()
            time.sleep(RATE_SLEEP)
            return resp.json()

        except Exception as e:
            if attempt == retries - 1:
                print(f"  ⚠️  Failed ({retries} attempts): {url}\n      {e}")
                return None
            time.sleep(2 ** attempt)

def jolpica_get_paged(url_base, data_key, inner_key, limit=100):
    """Fetch all pages of a paginated Jolpica endpoint.
    
    Pass limit=1000 for season-level bulk endpoints (results, qualifying,
    sprint) to pull an entire season in one or two calls instead of per-round.
    """
    items, offset = [], 0
    while True:
        data = jolpica_get(f"{url_base}?limit={limit}&offset={offset}")
        if not data:
            break
        mr    = data.get("MRData", {})
        total = int(mr.get("total", 0))
        batch = mr.get(data_key, {}).get(inner_key, [])
        items.extend(batch)
        offset += limit
        if offset >= total or not batch:
            break
    return items

# ── Build lookup dicts from existing raw_tables ───────────────────────────────
drivers_df      = raw_tables['drivers'].copy()
constructors_df = raw_tables['constructors'].copy()
circuits_df     = raw_tables['circuits'].copy()
status_df       = raw_tables['status'].copy()
races_df        = raw_tables['races'].copy()

# Coerce key columns to int safely (Kaggle uses \N for nulls, already cleaned)
def safe_int(series):
    return pd.to_numeric(series, errors='coerce').fillna(0).astype(int)

driver_ref_to_id      = dict(zip(drivers_df['driverRef'],      safe_int(drivers_df['driverId'])))
constructor_ref_to_id = dict(zip(constructors_df['constructorRef'], safe_int(constructors_df['constructorId'])))
circuit_ref_to_id     = dict(zip(circuits_df['circuitRef'],    safe_int(circuits_df['circuitId'])))
status_str_to_id      = dict(zip(status_df['status'],          safe_int(status_df['statusId'])))

# ── ID counters (pick up after existing maxes) ────────────────────────────────
next_ids = {
    'race':       int(safe_int(races_df['raceId']).max())                               + 1,
    'result':     int(safe_int(raw_tables['results']['resultId']).max())                + 1,
    'quali':      int(safe_int(raw_tables['qualifying']['qualifyId']).max())            + 1,
    'ds':         int(safe_int(raw_tables['driver_standings']['driverStandingsId']).max()) + 1,
    'cs':         int(safe_int(raw_tables['constructor_standings']['constructorStandingsId']).max()) + 1,
    'sprint':     int(safe_int(raw_tables['sprint_results']['resultId']).max())         + 1,
    'driver':     int(safe_int(drivers_df['driverId']).max())                           + 1,
    'constructor':int(safe_int(constructors_df['constructorId']).max())                 + 1,
    'status':     int(safe_int(status_df['statusId']).max())                            + 1,
}

def next_id(key):
    val = next_ids[key]
    next_ids[key] += 1
    return val

# ── Helper: get or create statusId ───────────────────────────────────────────
def get_status_id(status_str):
    if status_str not in status_str_to_id:
        sid = next_id('status')
        status_str_to_id[status_str] = sid
    return status_str_to_id[status_str]

# ── Helper: get or create driverId (handles 2025/2026 new entrants) ───────────
def get_driver_id(d_obj):
    """d_obj is a Jolpica Driver dict. Adds to drivers_df if new."""
    ref = d_obj['driverId']
    if ref not in driver_ref_to_id:
        did = next_id('driver')
        driver_ref_to_id[ref] = did
        new_row = {
            'driverId':    did,
            'driverRef':   ref,
            'number':      d_obj.get('permanentNumber', None),
            'code':        d_obj.get('code', None),
            'forename':    d_obj.get('givenName', ''),
            'surname':     d_obj.get('familyName', ''),
            'dob':         d_obj.get('dateOfBirth', None),
            'nationality': d_obj.get('nationality', None),
            'url':         d_obj.get('url', None),
        }
        global drivers_df
        drivers_df = pd.concat([drivers_df, pd.DataFrame([new_row])], ignore_index=True)
        print(f"  ➕ New driver: {d_obj.get('givenName','')} {d_obj.get('familyName','')} ({ref})")
    return driver_ref_to_id[ref]

# ── Helper: get or create constructorId ───────────────────────────────────────
def get_constructor_id(c_obj):
    """c_obj is a Jolpica Constructor dict."""
    ref = c_obj['constructorId']
    if ref not in constructor_ref_to_id:
        cid = next_id('constructor')
        constructor_ref_to_id[ref] = cid
        new_row = {
            'constructorId':  cid,
            'constructorRef': ref,
            'name':           c_obj.get('name', ref),
            'nationality':    c_obj.get('nationality', None),
            'url':            c_obj.get('url', None),
        }
        global constructors_df
        constructors_df = pd.concat([constructors_df, pd.DataFrame([new_row])], ignore_index=True)
        print(f"  ➕ New constructor: {c_obj.get('name', ref)} ({ref})")
    return constructor_ref_to_id[ref]

print("✅ Lookup tables built")
print(f"   Known drivers: {len(driver_ref_to_id)}  |  constructors: {len(constructor_ref_to_id)}  |  circuits: {len(circuit_ref_to_id)}")
print(f"   Kaggle data through year: {int(safe_int(races_df['year']).max())}")
print(f"   Next race ID: {next_ids['race']}")
print(f"\n   API call budget: ~200 req/hr — using bulk season endpoints")
print(f"   (≈3 bulk calls/year instead of 72 per-round calls)")


In [ ]:
# ── Accumulators ──────────────────────────────────────────────────────────────
new_races, new_results, new_quali    = [], [], []
new_driver_standings, new_ctor_standings, new_sprints = [], [], []

race_id_map = {}   # (year, round) → raceId

for year in JOLPICA_YEARS:
    print(f"\n{'─'*60}")
    print(f"  📅  {year} season")
    print(f"{'─'*60}")

    # ── Step 1: Race schedule (builds race_id_map) ────────────────────────────
    # limit=100 is fine here — max 24 races per season
    print(f"  Fetching race schedule...")
    races = jolpica_get_paged(f"{JOLPICA_BASE}/{year}", "RaceTable", "Races", limit=100)
    if not races:
        print(f"  ⚠️  No race data for {year} — season may not have started yet")
        continue

    print(f"  Found {len(races)} races")
    for race in races:
        yr   = int(race['season'])
        rnd  = int(race['round'])
        cref = race['Circuit']['circuitId']
        cid  = circuit_ref_to_id.get(cref, -1)
        if cid == -1:
            print(f"  ⚠️  Unknown circuit: {cref} ({race['raceName']}) — circuitId will be -1")

        race_id = next_id('race')
        race_id_map[(yr, rnd)] = race_id

        new_races.append({
            'raceId':     race_id,
            'year':       yr,
            'round':      rnd,
            'circuitId':  cid,
            'name':       race['raceName'],
            'date':       race.get('date'),
            'time':       race.get('time'),
            'url':        race.get('url'),
            'fp1_date':   race.get('FirstPractice',  {}).get('date'),
            'fp1_time':   race.get('FirstPractice',  {}).get('time'),
            'fp2_date':   race.get('SecondPractice', {}).get('date'),
            'fp2_time':   race.get('SecondPractice', {}).get('time'),
            'fp3_date':   race.get('ThirdPractice',  {}).get('date'),
            'fp3_time':   race.get('ThirdPractice',  {}).get('time'),
            'quali_date': race.get('Qualifying',     {}).get('date'),
            'quali_time': race.get('Qualifying',     {}).get('time'),
            'sprint_date':race.get('Sprint',         {}).get('date'),
            'sprint_time':race.get('Sprint',         {}).get('time'),
        })

    # ── Step 2: Bulk race results (entire season in 1-2 calls) ───────────────
    # limit=1000 pulls all ~480 results (24 races × ~20 drivers) in one shot,
    # eliminating 24 individual per-round calls.
    print(f"  Fetching all {year} race results (bulk)...")
    result_races = jolpica_get_paged(
        f"{JOLPICA_BASE}/{year}/results", "RaceTable", "Races", limit=1000
    )
    results_added = 0
    for race_block in result_races:
        rnd     = int(race_block['round'])
        race_id = race_id_map.get((year, rnd))
        if race_id is None:
            continue
        for r in race_block.get('Results', []):
            new_results.append({
                'resultId':        next_id('result'),
                'raceId':          race_id,
                'driverId':        get_driver_id(r['Driver']),
                'constructorId':   get_constructor_id(r['Constructor']),
                'number':          r.get('number'),
                'grid':            r.get('grid'),
                'position':        r.get('position'),
                'positionText':    r.get('positionText'),
                'positionOrder':   r.get('position'),
                'points':          r.get('points'),
                'laps':            r.get('laps'),
                'time':            r.get('Time', {}).get('time'),
                'milliseconds':    r.get('Time', {}).get('millis'),
                'fastestLap':      r.get('FastestLap', {}).get('lap'),
                'rank':            r.get('FastestLap', {}).get('rank'),
                'fastestLapTime':  r.get('FastestLap', {}).get('Time', {}).get('time'),
                'fastestLapSpeed': r.get('FastestLap', {}).get('AverageSpeed', {}).get('speed'),
                'statusId':        get_status_id(r.get('status', 'Unknown')),
            })
            results_added += 1
    print(f"  ✅ {results_added} race results")

    # ── Step 3: Bulk qualifying (entire season in 1-2 calls) ─────────────────
    print(f"  Fetching all {year} qualifying (bulk)...")
    quali_races = jolpica_get_paged(
        f"{JOLPICA_BASE}/{year}/qualifying", "RaceTable", "Races", limit=1000
    )
    quali_added = 0
    for race_block in quali_races:
        rnd     = int(race_block['round'])
        race_id = race_id_map.get((year, rnd))
        if race_id is None:
            continue
        for q in race_block.get('QualifyingResults', []):
            new_quali.append({
                'qualifyId':     next_id('quali'),
                'raceId':        race_id,
                'driverId':      get_driver_id(q['Driver']),
                'constructorId': get_constructor_id(q['Constructor']),
                'number':        q.get('number'),
                'position':      q.get('position'),
                'q1':            q.get('Q1'),
                'q2':            q.get('Q2'),
                'q3':            q.get('Q3'),
            })
            quali_added += 1
    print(f"  ✅ {quali_added} qualifying entries")

    # ── Step 4: Bulk sprint results (entire season in 1 call) ────────────────
    # Sprint races exist only at 6–8 rounds per season; non-sprint 404s are
    # handled gracefully by jolpica_get.
    print(f"  Fetching all {year} sprint results (bulk)...")
    sprint_races = jolpica_get_paged(
        f"{JOLPICA_BASE}/{year}/sprint", "RaceTable", "Races", limit=1000
    )
    sprint_added = 0
    for race_block in sprint_races:
        rnd     = int(race_block['round'])
        race_id = race_id_map.get((year, rnd))
        if race_id is None:
            continue
        for s in race_block.get('SprintResults', []):
            new_sprints.append({
                'resultId':      next_id('sprint'),
                'raceId':        race_id,
                'driverId':      get_driver_id(s['Driver']),
                'constructorId': get_constructor_id(s['Constructor']),
                'number':        s.get('number'),
                'grid':          s.get('grid'),
                'position':      s.get('position'),
                'positionText':  s.get('positionText'),
                'positionOrder': s.get('position'),
                'points':        s.get('points'),
                'laps':          s.get('laps'),
                'time':          s.get('Time', {}).get('time'),
                'milliseconds':  s.get('Time', {}).get('millis'),
                'fastestLap':    s.get('FastestLap', {}).get('lap'),
                'fastestLapTime':s.get('FastestLap', {}).get('Time', {}).get('time'),
                'statusId':      get_status_id(s.get('status', 'Unknown')),
            })
            sprint_added += 1
    print(f"  ✅ {sprint_added} sprint results")

    # ── Step 5: Driver standings (one snapshot per round, paginated) ──────────
    # /{year}/driverStandings returns all round snapshots for the season.
    # With ~20 drivers × 24 rounds = ~480 entries, limit=500 pulls it in 1 call.
    print(f"  Fetching driver standings...")
    ds_all = jolpica_get_paged(
        f"{JOLPICA_BASE}/{year}/driverStandings",
        "StandingsTable", "StandingsLists",
        limit=500
    )
    ds_added = 0
    for standing_list in ds_all:
        yr_s = int(standing_list['season'])
        rnd  = int(standing_list['round'])
        rid  = race_id_map.get((yr_s, rnd))
        if rid is None:
            continue
        for s in standing_list.get('DriverStandings', []):
            new_driver_standings.append({
                'driverStandingsId': next_id('ds'),
                'raceId':            rid,
                'driverId':          driver_ref_to_id.get(s['Driver']['driverId'], -1),
                'points':            s.get('points'),
                'position':          s.get('position'),
                'positionText':      s.get('positionText'),
                'wins':              s.get('wins'),
            })
            ds_added += 1
    print(f"  ✅ {ds_added} driver standing entries")

    # ── Step 6: Constructor standings ─────────────────────────────────────────
    # ~10 constructors × 24 rounds = ~240 entries, fits in 1 call.
    print(f"  Fetching constructor standings...")
    cs_all = jolpica_get_paged(
        f"{JOLPICA_BASE}/{year}/constructorStandings",
        "StandingsTable", "StandingsLists",
        limit=500
    )
    cs_added = 0
    for standing_list in cs_all:
        yr_s = int(standing_list['season'])
        rnd  = int(standing_list['round'])
        rid  = race_id_map.get((yr_s, rnd))
        if rid is None:
            continue
        for s in standing_list.get('ConstructorStandings', []):
            new_ctor_standings.append({
                'constructorStandingsId': next_id('cs'),
                'raceId':                 rid,
                'constructorId':          constructor_ref_to_id.get(
                                            s['Constructor']['constructorId'], -1),
                'points':                 s.get('points'),
                'position':               s.get('position'),
                'positionText':           s.get('positionText'),
                'wins':                   s.get('wins'),
            })
            cs_added += 1
    print(f"  ✅ {cs_added} constructor standing entries")

print(f"\n{'='*60}")
print(f"✅ Jolpica fetch complete")
print(f"   Races:                 {len(new_races)}")
print(f"   Results:               {len(new_results)}")
print(f"   Qualifying:            {len(new_quali)}")
print(f"   Sprint results:        {len(new_sprints)}")
print(f"   Driver standings:      {len(new_driver_standings)}")
print(f"   Constructor standings: {len(new_ctor_standings)}")
print(f"   New drivers added:     {len(drivers_df) - len(raw_tables['drivers'])}")
print(f"   New constructors added:{len(constructors_df) - len(raw_tables['constructors'])}")


In [ ]:
# ── Build new-data DataFrames ─────────────────────────────────────────────────
new_status_rows = [
    {'statusId': sid, 'status': s}
    for s, sid in status_str_to_id.items()
    if sid not in set(safe_int(raw_tables['status']['statusId']))
]

# ── Merge into raw_tables (concat + reset index) ──────────────────────────────
def merge_into(table_key, new_rows, id_col):
    if not new_rows:
        print(f"  ℹ️  {table_key}: no new rows to add")
        return
    new_df  = pd.DataFrame(new_rows)
    merged  = pd.concat([raw_tables[table_key], new_df], ignore_index=True)
    # Dedup on primary key — safe to re-run
    merged  = merged.drop_duplicates(subset=[id_col], keep='last')
    raw_tables[table_key] = merged
    print(f"  ✅ {table_key:<28} {len(raw_tables[table_key]):>7,} rows  (+{len(new_rows):,} new)")

print("Merging into raw_tables...\n")
merge_into('races',                 new_races,             'raceId')
merge_into('results',               new_results,           'resultId')
merge_into('qualifying',            new_quali,             'qualifyId')
merge_into('sprint_results',        new_sprints,           'resultId')
merge_into('driver_standings',      new_driver_standings,  'driverStandingsId')
merge_into('constructor_standings', new_ctor_standings,    'constructorStandingsId')
merge_into('status',                new_status_rows,       'statusId')

# Replace drivers/constructors with the enriched versions (may have new entrants)
old_d = len(raw_tables['drivers'])
old_c = len(raw_tables['constructors'])
raw_tables['drivers']      = drivers_df.drop_duplicates(subset=['driverId'], keep='last').reset_index(drop=True)
raw_tables['constructors'] = constructors_df.drop_duplicates(subset=['constructorId'], keep='last').reset_index(drop=True)
print(f"  ✅ {'drivers':<28} {len(raw_tables['drivers']):>7,} rows  (+{len(raw_tables['drivers'])-old_d:,} new)")
print(f"  ✅ {'constructors':<28} {len(raw_tables['constructors']):>7,} rows  (+{len(raw_tables['constructors'])-old_c:,} new)")

print(f"\n✅ raw_tables updated. Section 2 will load everything to BigQuery.")
print(f"   Year range in races: {int(safe_int(raw_tables['races']['year']).min())} – {int(safe_int(raw_tables['races']['year']).max())}")

---
## Section 2: F1 Race Data — Clean & Stage to GCS (Parquet)

The raw Kaggle CSVs use `\N` as a null placeholder (the MySQL dump convention from Ergast), and several columns that look like numbers are stored as strings. This section standardises the data and writes all 15 tables to GCS as Parquet files.

**Why Parquet?**
- Type fidelity: numeric columns stay numeric, nulls stay null — no schema guessing when students load to BigQuery
- Storage efficient: `lap_times` (589 K rows) compresses from ~28 MB CSV to ~8 MB Parquet
- Direct BigQuery compatibility: `bq load --source_format=PARQUET` just works

**Bucket layout:**
```
gs://class-demo/mclaren-f1/
  bq-staging/
    circuits.parquet
    constructor_results.parquet
    ... (all 15 tables)
```

Students will run a single `bq load` command per table (or a loop) as the first step of the lab's data engineering section. The data is ready to query the moment the load finishes — no additional cleaning needed.


In [16]:
# Clean all tables: replace Ergast null sentinels, infer correct dtypes,
# and standardize column names to snake_case.
import numpy as np
import re

def clean_table(df: pd.DataFrame, table_name: str) -> pd.DataFrame:
    """Replace \\N nulls, fix numeric columns, clean column names."""
    df = df.copy()

    # Normalize column names to snake_case
    def to_snake(name):
        s1 = re.sub(r'(.)([A-Z][a-z]+)', r'\1_\2', name)
        return re.sub(r'([a-z0-9])([A-Z])', r'\1_\2', s1).lower()
    df.columns = [to_snake(c) for c in df.columns]

    # Replace Ergast null sentinel with NaN
    df.replace(r'^\\N$', np.nan, regex=True, inplace=True)

    # Coerce obviously numeric columns that arrived as object/string
    for col in df.columns:
        if df[col].dtype == object:
            try:
                converted = pd.to_numeric(df[col], errors='raise')
                # Only coerce if the column name suggests it's numeric
                numeric_hints = [
                    'id', 'year', 'round', 'grid', 'position', 'points',
                    'laps', 'rank', 'number', 'milliseconds', 'fastestlap',
                    'stop', 'lap', 'duration', 'wins', 'altitude', 'lat', 'lng'
                ]
                if any(hint in col.lower() for hint in numeric_hints):
                    df[col] = pd.to_numeric(df[col], errors='coerce')
            except (ValueError, TypeError):
                pass

    return df


cleaned_tables = {}
for name, df in raw_tables.items():
    cleaned_tables[name] = clean_table(df, name)

print("✅ All tables cleaned")
print("\nSample — drivers columns after cleaning:")
print(cleaned_tables["drivers"].dtypes)

✅ All tables cleaned

Sample — drivers columns after cleaning:
driver_id        int64
driver_ref      object
number         float64
code            object
forename        object
surname         object
dob             object
nationality     object
url             object
dtype: object


In [17]:
# Build the mclaren_drivers table.
# This joins driver info with their McLaren career stats: first year,
# last year, total races, wins, podiums, and points scored with McLaren.
# The lab's agent and profile generation both reference this table.

results_clean      = cleaned_tables["results"]
races_clean        = cleaned_tables["races"]
drivers_clean      = cleaned_tables["drivers"]
constructors_clean = cleaned_tables["constructors"]

# Re-derive McLaren ID from cleaned table (column is now snake_case)
mclaren_id = constructors_clean[
    constructors_clean["constructor_ref"] == MCLAREN_REF
].iloc[0]["constructor_id"]

mac_results = results_clean[results_clean["constructor_id"] == mclaren_id].copy()
mac_results = mac_results.merge(races_clean[["race_id", "year", "round", "name", "circuit_id"]], on="race_id")

# Aggregate career stats per driver with McLaren
def safe_pos(series):
    """Count rows where position is numeric and <= threshold."""
    numeric = pd.to_numeric(series, errors="coerce")
    return numeric

driver_stats = mac_results.groupby("driver_id").agg(
    mclaren_first_year=("year", "min"),
    mclaren_last_year=("year", "max"),
    mclaren_races=("race_id", "count"),
    mclaren_points=("points", "sum"),
).reset_index()

# Count wins (position == 1) and podiums (position <= 3)
pos_numeric = pd.to_numeric(mac_results["position_order"], errors="coerce") \
    if "position_order" in mac_results.columns \
    else pd.to_numeric(mac_results.get("position", pd.Series(dtype=float)), errors="coerce")

mac_results["_pos"] = pos_numeric
wins_df   = mac_results[mac_results["_pos"] == 1].groupby("driver_id").size().reset_index(name="mclaren_wins")
podium_df = mac_results[mac_results["_pos"] <= 3].groupby("driver_id").size().reset_index(name="mclaren_podiums")

driver_stats = driver_stats.merge(wins_df,   on="driver_id", how="left")
driver_stats = driver_stats.merge(podium_df, on="driver_id", how="left")
driver_stats[["mclaren_wins", "mclaren_podiums"]] = \
    driver_stats[["mclaren_wins", "mclaren_podiums"]].fillna(0).astype(int)

# Join with driver biographical info
mclaren_drivers = drivers_clean[
    drivers_clean["driver_id"].isin(driver_stats["driver_id"])
].merge(driver_stats, on="driver_id")

# Sort by first year with McLaren for readability
mclaren_drivers = mclaren_drivers.sort_values("mclaren_first_year").reset_index(drop=True)

print(f"✅ mclaren_drivers table built: {len(mclaren_drivers)} drivers")
print()
print(mclaren_drivers[[
    "forename", "surname", "mclaren_first_year", "mclaren_last_year",
    "mclaren_races", "mclaren_wins", "mclaren_podiums", "mclaren_points"
]].to_string(index=False))

✅ mclaren_drivers table built: 55 drivers

 forename       surname  mclaren_first_year  mclaren_last_year  mclaren_races  mclaren_wins  mclaren_podiums  mclaren_points
    Denny         Hulme                1968               1974             53             3               12            94.0
    Bruce       McLaren                1968               1968              1             0                0             0.0
   Jackie        Oliver                1971               1971              3             0                0             0.0
       Jo       Bonnier                1971               1971              5             0                0             0.0
    Peter        Gethin                1971               1971              7             0                0             0.0
     Mark       Donohue                1971               1971              1             0                1             4.0
    David         Hobbs                1971               1974              3     

In [ ]:
# Stage all cleaned tables to GCS as Parquet files.
#
# 15 tables total (14 Ergast + mclaren_drivers), written to:
#   gs://class-demo/mclaren-f1/bq-staging/<table_name>.parquet
#
# In the lab, students load these into BigQuery with:
#   bq load --source_format=PARQUET \
#     ${PROJECT_ID}:f1_data.<table_name> \
#     gs://class-demo/mclaren-f1/bq-staging/<table_name>.parquet

import io
import pyarrow as pa
import pyarrow.parquet as pq

stage_manifest = {**cleaned_tables, "mclaren_drivers": mclaren_drivers}

bucket = gcs_client.bucket(GCS_BUCKET)
parquet_gcs_paths = []

print(f"Staging {len(stage_manifest)} tables to gs://{GCS_BUCKET}/{GCS_BQ_STAGING}/\n")
print(f"  {'Table':<30} {'Rows':>8}  {'Size':>8}  GCS path")
print(f"  {'-'*72}")

for table_name, df in stage_manifest.items():
    blob_name = f"{GCS_BQ_STAGING}/{table_name}.parquet"
    blob = bucket.blob(blob_name)

    # Serialise to in-memory Parquet buffer, then stream to GCS
    buf = io.BytesIO()
    df.to_parquet(buf, index=False, engine="pyarrow", compression="snappy")
    size_kb = buf.tell() / 1024
    buf.seek(0)
    blob.upload_from_file(buf, content_type="application/octet-stream")

    gcs_uri = f"gs://{GCS_BUCKET}/{blob_name}"
    parquet_gcs_paths.append(gcs_uri)
    print(f"  ✅ {table_name:<30} {len(df):>8,}  {size_kb:>6.0f} KB  {blob_name}")

print(f"\n✅ {len(parquet_gcs_paths)} tables staged to GCS")
print(f"\nBigQuery load snippet (run once in the lab):")
print(f"  for table in $(gsutil ls gs://{GCS_BUCKET}/{GCS_BQ_STAGING}/*.parquet); do")
print(f"    name=$(basename $table .parquet)")
print(f"    bq load --source_format=PARQUET ${{PROJECT_ID}}:f1_data.$name $table")
print(f"  done")


---
## Section 3: FIA Regulation Documents

The FIA publishes its official Formula 1 regulations on [fia.com](https://www.fia.com). These PDFs form the RAG document store for the Gemini Enterprise tier of the lab — when users ask regulation questions, the agent retrieves answers from these documents.

We download the most current versions available, validate them, and stage them to GCS.

**Documents we want:**
- FIA Formula 1 Sporting Regulations (~120 pages)
- FIA Formula 1 Technical Regulations (~90 pages)
- Appendix L / International Sporting Code (driving standards)
- FIA Financial Regulations (cost cap rules — increasingly important)

In [21]:
# FIA document URLs.
# The FIA posts regulations at predictable paths on their website.
# These URLs point to the 2025 season documents. If a URL returns 404,
# visit https://www.fia.com/regulation/category/110 and grab the
# current PDF links manually, then update the dict below.

import requests
import time
from pathlib import Path

FIA_DOCS = {
    "fia_f1_sporting_regulations_2025.pdf": (
        "https://api.fia.com/system/files/documents/"
        "fia_2025_formula_1_sporting_regulations_-_issue_5_-_2025-04-30.pdf"
    ),
    "fia_f1_technical_regulations_2025.pdf": (
        "https://www.fia.com/sites/default/files/documents/"
        "fia_2025_formula_1_technical_regulations_-_issue_03_-_2025-04-07.pdf"
    ),
    "fia_f1_financial_regulations_2025.pdf": (
        "https://api.fia.com/system/files/documents/"
        "fia_formula_1_financial_regulations_-_issue_24_-_2025-04-30_0.pdf"
    ),
    "fia_appendix_l_2025.pdf": (
        "https://www.fia.com/system/files/documents/"
        "appendix_l_2025_publie_le_10_decembre_2025_2.pdf"
    ),
    "fia_f1_driving_standards_guidelines_2025.pdf": (
        "https://www.fia.com/sites/default/files/"
        "f1_driving_standards_guidelines_version_4.1_feb_20_2025.pdf"
    ),
}

# Download headers to appear as a browser request
HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

fia_dir = Path("/tmp/fia_docs")
fia_dir.mkdir(exist_ok=True)

downloaded = {}
failed = {}

for filename, url in FIA_DOCS.items():
    local_path = fia_dir / filename
    print(f"Downloading {filename}...")
    try:
        resp = requests.get(url, headers=HEADERS, timeout=60, stream=True)
        if resp.status_code == 200 and resp.headers.get("content-type", "").startswith("application/pdf"):
            with open(local_path, "wb") as f:
                for chunk in resp.iter_content(chunk_size=8192):
                    f.write(chunk)
            size_kb = local_path.stat().st_size / 1024
            # Sanity check: real PDFs start with %PDF
            with open(local_path, "rb") as f:
                header = f.read(4)
            if header == b"%PDF":
                print(f"  ✅ {size_kb:.0f} KB — valid PDF")
                downloaded[filename] = local_path
            else:
                print(f"  ⚠️  Downloaded but doesn't look like a PDF (got {header})")
                failed[filename] = f"Not a PDF (HTTP {resp.status_code})"
        else:
            print(f"  ❌ HTTP {resp.status_code} — URL may have changed")
            failed[filename] = f"HTTP {resp.status_code}"
    except Exception as e:
        print(f"  ❌ {e}")
        failed[filename] = str(e)
    time.sleep(1)  # Be polite to the FIA server

print(f"\n✅ Downloaded: {len(downloaded)} | ❌ Failed: {len(failed)}")
if failed:
    print("\nFailed downloads — please retrieve these manually from https://www.fia.com/regulation/category/110")
    print("and place them in /tmp/fia_docs/ before running the GCS upload cell:")
    for fn, reason in failed.items():
        print(f"  {fn}: {reason}")

  ✅ 1264 KB — valid PDF
  ✅ 2190 KB — valid PDF
  ✅ 306 KB — valid PDF
  ✅ 2481 KB — valid PDF
  ✅ 422 KB — valid PDF

✅ Downloaded: 5 | ❌ Failed: 0


In [22]:
# Upload FIA PDFs to GCS.
# All PDFs found in /tmp/fia_docs/ are uploaded, whether auto-downloaded
# or placed there manually.

bucket = gcs_client.bucket(GCS_BUCKET)
all_pdfs = list(fia_dir.glob("*.pdf"))

print(f"Uploading {len(all_pdfs)} PDF(s) to gs://{GCS_BUCKET}/{GCS_FIA_PREFIX}/...\n")

fia_gcs_paths = []
for pdf_path in all_pdfs:
    blob_name = f"{GCS_FIA_PREFIX}/{pdf_path.name}"
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(str(pdf_path), content_type="application/pdf")
    gcs_uri = f"gs://{GCS_BUCKET}/{blob_name}"
    fia_gcs_paths.append(gcs_uri)
    size_kb = pdf_path.stat().st_size / 1024
    print(f"  ✅ {pdf_path.name} ({size_kb:.0f} KB) → {gcs_uri}")

print(f"\n✅ FIA documents staged to GCS")

Uploading 5 PDF(s) to gs://class-demo/mclaren-f1/fia-docs/...

  ✅ fia_f1_driving_standards_guidelines_2025.pdf (422 KB) → gs://class-demo/mclaren-f1/fia-docs/fia_f1_driving_standards_guidelines_2025.pdf
  ✅ fia_f1_technical_regulations_2025.pdf (2190 KB) → gs://class-demo/mclaren-f1/fia-docs/fia_f1_technical_regulations_2025.pdf
  ✅ fia_appendix_l_2025.pdf (2481 KB) → gs://class-demo/mclaren-f1/fia-docs/fia_appendix_l_2025.pdf
  ✅ fia_f1_sporting_regulations_2025.pdf (1264 KB) → gs://class-demo/mclaren-f1/fia-docs/fia_f1_sporting_regulations_2025.pdf
  ✅ fia_f1_financial_regulations_2025.pdf (306 KB) → gs://class-demo/mclaren-f1/fia-docs/fia_f1_financial_regulations_2025.pdf

✅ FIA documents staged to GCS


---
## Section 4: McLaren Driver & Team Profiles

This section generates natural-language profiles for every driver who has raced for McLaren, plus a team profile. Profiles are used in the Gemini Enterprise structured data store — when the agent answers questions like "Tell me about Ayrton Senna's time at McLaren," it retrieves from these profiles.

**Profile design:**
- 2–3 paragraphs per driver
- Covers their McLaren chapter specifically: career context, seasons at McLaren, key results, racing style, notable moments
- Does *not* cover post-McLaren career (brand sensitivity requirement)
- Generated by Gemini with Google Search grounding for factual accuracy

**Embeddings:**
- 3072-dimensional vectors (gemini-embedding-001 maximum)
- Pre-computed here so the lab doesn't need to generate them at runtime
- Enables semantic similarity search across the profile store

Uses the same parallel threading pattern as the movie summary notebook — 20 workers, progress saved incrementally.

In [ ]:
# Initialize the Gemini client for Vertex AI.
# google.auth.default() was already called above; we reuse the same
# project and credentials.
import threading
from google import genai
from google.genai import types

_thread_local = threading.local()

def get_gemini_client():
    """Return a thread-local Gemini client (safe for parallel calls)."""
    if getattr(_thread_local, "client", None) is None:
        _thread_local.client = genai.Client(
            vertexai=True,
            project=PROJECT_ID,
            location="us-central1",
        )
    return _thread_local.client

# Quick connectivity test
test_resp = get_gemini_client().models.generate_content(
    model=GEMINI_MODEL,
    contents="In one sentence, what is McLaren Racing?",
    config=types.GenerateContentConfig(max_output_tokens=100)
)
print(f"✅ Gemini client ready")
print(f"   Test response: {test_resp.text.strip()}")

In [ ]:
# Build the context payload for each McLaren driver.
# We enrich the prompt with structured race data from our dataset:
# seasons raced, race count, wins, podiums, and a sample of notable results.
# This grounds the generation in actual data rather than just model knowledge.

def build_driver_context(driver_row: pd.Series, mac_results_df: pd.DataFrame,
                          races_df: pd.DataFrame) -> dict:
    """Build a structured context dict for a McLaren driver."""
    did = driver_row["driver_id"]
    full_name = f"{driver_row['forename']} {driver_row['surname']}"

    drv_results = mac_results_df[mac_results_df["driver_id"] == did].copy()
    drv_results = drv_results.merge(races_df[["race_id", "year", "name", "round"]], on="race_id")

    seasons = sorted(drv_results["year"].unique().tolist())
    pos_num = pd.to_numeric(drv_results.get("position_order",
                            drv_results.get("position", pd.Series(dtype=float))),
                            errors="coerce")
    wins    = int((pos_num == 1).sum())
    podiums = int((pos_num <= 3).sum())
    points  = float(drv_results["points"].sum()) if "points" in drv_results else 0.0

    # Top results for the prompt
    top = drv_results.copy()
    top["_pos"] = pos_num
    top = top[top["_pos"] <= 10].sort_values(["_pos", "year"]).head(8)
    notable = [
        f"P{int(r['_pos'])} — {r['name']} {int(r['year'])}"
        for _, r in top.iterrows()
    ]

    return {
        "driver_id": int(did),
        "full_name": full_name,
        "forename": driver_row["forename"],
        "surname": driver_row["surname"],
        "nationality": driver_row.get("nationality", ""),
        "dob": str(driver_row.get("dob", "")),
        "seasons_at_mclaren": seasons,
        "mclaren_races": int(len(drv_results)),
        "mclaren_wins": wins,
        "mclaren_podiums": podiums,
        "mclaren_points": round(points, 1),
        "notable_results": notable,
    }


# Build context for all McLaren drivers
mac_results_clean = results_clean[results_clean["constructor_id"] == mclaren_id].copy()
races_clean_sub   = races_clean[["race_id", "year", "name", "round"]]

driver_contexts = []
for _, row in mclaren_drivers.iterrows():
    ctx = build_driver_context(row, mac_results_clean, races_clean)
    driver_contexts.append(ctx)

print(f"✅ Built context for {len(driver_contexts)} drivers")
print("\nSample context (first driver):")
import json
print(json.dumps(driver_contexts[0], indent=2))

In [ ]:
# Profile generation prompt and function.
#
# Key design choices:
#  - Google Search grounding is enabled so Gemini can verify facts
#  - We explicitly instruct it to cover only the McLaren chapter
#  - Race data from our dataset is injected so output is grounded in reality
#  - Profile length scales with significance: more races = richer context

def build_profile_prompt(ctx: dict) -> str:
    """Build a prompt that generates a McLaren-chapter driver profile."""
    seasons_str = ", ".join(str(y) for y in ctx["seasons_at_mclaren"])
    notable_str = "\n".join(f"  - {r}" for r in ctx["notable_results"]) \
                  if ctx["notable_results"] else "  - No top-10 finishes in dataset"

    return f"""Write a factual 2–3 paragraph profile of **{ctx['full_name']}** focused \
specifically on their time racing for the McLaren Formula 1 team.

IMPORTANT CONSTRAINTS:
- Cover their McLaren chapter only. Mention their pre-McLaren career briefly (1–2 sentences) \
for context, but do NOT discuss what they did after leaving McLaren.
- Write in a neutral, encyclopedic tone suitable for a technical reference document.
- Be factual and specific. Include seasons, key races, championship results where relevant.
- Keep total length to 200–350 words.

KNOWN DATA FROM THE DATASET:
- Nationality: {ctx['nationality']}
- Seasons at McLaren: {seasons_str}
- Races with McLaren: {ctx['mclaren_races']}
- Wins with McLaren: {ctx['mclaren_wins']}
- Podiums with McLaren: {ctx['mclaren_podiums']}
- Points with McLaren: {ctx['mclaren_points']}
- Notable results (from race data):
{notable_str}

Use these data points as anchors. You may use search to fill in additional accurate details \
such as championship positions, specific race names, or career significance."""


def generate_driver_profile(
    ctx: dict,
    max_retries: int = 3,
    retry_delay: float = 2.0
) -> tuple[str | None, str | None]:
    """Generate a profile for one driver. Returns (profile_text, error)."""
    prompt = build_profile_prompt(ctx)
    gen_config = types.GenerateContentConfig(
        temperature=0.7,
        tools=[types.Tool(google_search=types.GoogleSearch())],
    )
    for attempt in range(max_retries):
        try:
            resp = get_gemini_client().models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=gen_config,
            )
            text = resp.text.strip() if resp.text else None
            if text and len(text) > 80:
                return text, None
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay * (attempt + 1))
            else:
                return None, f"{type(e).__name__}: {e}"
    return None, "Max retries exceeded or response too short"


print("✅ Profile generation functions defined")
print("\nTesting with Emerson Fittipaldi context...")
# Quick test with a single driver
test_ctx = next((c for c in driver_contexts if "Fittipaldi" in c["surname"]),
                driver_contexts[0])
test_text, test_err = generate_driver_profile(test_ctx)
if test_text:
    print(f"✅ Test profile generated ({len(test_text)} chars)")
    print(f"Preview: {test_text[:300]}...")
else:
    print(f"❌ Test failed: {test_err}")

In [ ]:
# Generate profiles for all McLaren drivers in parallel.
# Uses 20 workers (driver count is small enough that more isn't needed).
# Progress is saved every 10 profiles so interruptions don't lose work.

import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.notebook import tqdm

MAX_WORKERS    = 20
SAVE_INTERVAL  = 10
PROFILE_FILE   = "/tmp/mclaren_driver_profiles.json"
PROFILE_PROG   = "/tmp/mclaren_profile_progress.json"


def load_progress() -> dict:
    """Load previously generated profiles from the progress file."""
    prog_path = Path(PROFILE_PROG)
    if prog_path.exists():
        with open(prog_path) as f:
            data = json.load(f)
        print(f"📂 Resuming from progress file: {len(data)} profiles already done")
        return {int(k): v for k, v in data.items()}
    return {}


def save_progress(profiles: dict):
    with open(PROFILE_PROG, "w") as f:
        json.dump({str(k): v for k, v in profiles.items()}, f, indent=2, ensure_ascii=False)


def process_one_driver(ctx: dict) -> dict:
    """Worker function: generate profile for one driver context."""
    text, err = generate_driver_profile(ctx)
    return {
        "driver_id": ctx["driver_id"],
        "full_name": ctx["full_name"],
        "metadata": {
            "nationality": ctx["nationality"],
            "dob": ctx["dob"],
            "seasons_at_mclaren": ctx["seasons_at_mclaren"],
            "mclaren_races": ctx["mclaren_races"],
            "mclaren_wins": ctx["mclaren_wins"],
            "mclaren_podiums": ctx["mclaren_podiums"],
            "mclaren_points": ctx["mclaren_points"],
        },
        "profile_text": text,
        "profile_error": err,
    }


# Load any existing progress
profiles_done = load_progress()   # {driver_id: profile_dict}
remaining = [c for c in driver_contexts if c["driver_id"] not in profiles_done]

print(f"Total drivers  : {len(driver_contexts)}")
print(f"Already done   : {len(profiles_done)}")
print(f"Remaining      : {len(remaining)}")
print()

if remaining:
    with tqdm(total=len(remaining), desc="Driver profiles") as pbar:
        with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
            futures = {executor.submit(process_one_driver, ctx): ctx for ctx in remaining}
            for future in as_completed(futures):
                result = future.result()
                profiles_done[result["driver_id"]] = result
                pbar.update(1)
                if len(profiles_done) % SAVE_INTERVAL == 0:
                    save_progress(profiles_done)

    save_progress(profiles_done)

errors = [p for p in profiles_done.values() if p.get("profile_error")]
success = len(profiles_done) - len(errors)
print(f"\n✅ Profiles: {success} generated, {len(errors)} errors")
if errors:
    for e in errors:
        print(f"  ❌ {e['full_name']}: {e['profile_error']}")

In [ ]:
# Generate the McLaren team/constructor profile.
# This is a single profile covering the team's full history:
# founding, key eras, championship record, and current standing.
# Same prompt design as driver profiles — McLaren-chapter framing.

# Build team stats from our data
mac_con_standings = cleaned_tables.get("constructor_standings", pd.DataFrame())
if not mac_con_standings.empty:
    mac_standings = mac_con_standings[
        mac_con_standings["constructor_id"] == mclaren_id
    ].merge(races_clean[["race_id", "year"]], on="race_id", how="left")
    championship_years = mac_standings[mac_standings["position"] == 1]["year"].unique().tolist()
else:
    championship_years = []

total_mac_wins   = int((pd.to_numeric(mac_results_clean.get(
    "position_order", mac_results_clean.get("position", pd.Series())), errors="coerce") == 1).sum())
total_mac_races  = mac_results_clean["race_id"].nunique()

team_prompt = f"""Write a factual 3–4 paragraph profile of the **McLaren Formula 1 team**.

Cover:
1. Founding and early history (Bruce McLaren era)
2. Championship eras — including the dominant 1980s–90s period with Senna and Prost
3. Post-2000 performance through to the 2024 season
4. The team's identity, culture, and engineering philosophy

KNOWN DATA FROM DATASET:
- Total race entries: {total_mac_races:,}
- Total race wins: {total_mac_wins:,}
- Constructor championship seasons (from data): {sorted(championship_years)}

Tone: neutral and encyclopedic, suitable for a technical reference document.
Length: 300–450 words.
Use search to verify specific championship years, driver rosters, and historical details."""

team_gen_config = types.GenerateContentConfig(
    temperature=0.6,
    tools=[types.Tool(google_search=types.GoogleSearch())],
)
team_resp = get_gemini_client().models.generate_content(
    model=GEMINI_MODEL,
    contents=team_prompt,
    config=team_gen_config,
)
team_profile_text = team_resp.text.strip()

print(f"✅ Team profile generated ({len(team_profile_text)} chars)")
print(f"\nPreview:\n{team_profile_text[:400]}...")

In [ ]:
# Generate 3072-dimensional embeddings for all driver profiles and the team profile.
# Pre-computing these here means the lab doesn't generate embeddings at runtime.
# Same parallel threading pattern as the movie notebook.

def generate_embedding(text: str, label: str,
                       max_retries: int = 3,
                       retry_delay: float = 2.0) -> tuple[list | None, str | None]:
    """Generate a 3072-dim embedding for a profile text string."""
    for attempt in range(max_retries):
        try:
            resp = get_gemini_client().models.embed_content(
                model=EMBED_MODEL,
                contents=text,
                config=types.EmbedContentConfig(output_dimensionality=EMBED_DIM)
            )
            if resp.embeddings and resp.embeddings[0].values:
                return list(resp.embeddings[0].values), None
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(retry_delay * (attempt + 1))
            else:
                return None, f"{type(e).__name__}: {e}"
    return None, "No embedding returned after retries"


def embed_one(item: dict) -> dict:
    """Worker: embed a profile item's text."""
    text = item.get("profile_text", "")
    if not text:
        return {**item, "embedding": None, "embedding_error": "No profile text"}
    vec, err = generate_embedding(text, item.get("full_name", ""))
    return {**item, "embedding": vec, "embedding_error": err}


# Collect all items to embed: driver profiles + team profile
embed_inputs = [
    p for p in profiles_done.values() if p.get("profile_text")
]
team_item = {
    "driver_id": -1,
    "full_name": "McLaren Racing (Constructor)",
    "metadata": {"type": "constructor", "constructor_ref": MCLAREN_REF},
    "profile_text": team_profile_text,
}
embed_inputs.append(team_item)

print(f"Generating embeddings for {len(embed_inputs)} profiles ({EMBED_DIM} dimensions each)...")

embedded_profiles = []
with tqdm(total=len(embed_inputs), desc="Embeddings") as pbar:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(embed_one, item): item for item in embed_inputs}
        for future in as_completed(futures):
            embedded_profiles.append(future.result())
            pbar.update(1)

embed_ok  = [p for p in embedded_profiles if p.get("embedding")]
embed_err = [p for p in embedded_profiles if not p.get("embedding")]

print(f"\n✅ Embeddings: {len(embed_ok)} generated, {len(embed_err)} errors")
if embed_err:
    for e in embed_err:
        print(f"  ❌ {e['full_name']}: {e.get('embedding_error')}")

# Verify dimensionality
if embed_ok:
    sample_dim = len(embed_ok[0]["embedding"])
    print(f"\nEmbedding dimension check: {sample_dim} (expected {EMBED_DIM}) "
          f"{'✅' if sample_dim == EMBED_DIM else '❌'}")

In [ ]:
# Write the final profiles JSON (with embeddings) and upload to GCS.
#
# Two files are written:
#   1. mclaren_profiles_with_embeddings.json  — full file (large — for vector store / AlloyDB import)
#   2. mclaren_profiles.json                  — profiles without embeddings (for review)
#
# The JSON structure for each entry:
#   {
#     "driver_id": int,
#     "full_name": str,
#     "metadata": { seasons, races, wins, ... },
#     "profile_text": str,
#     "embedding": [float, ...]   # 3072 values
#   }

PROFILES_FULL  = "/tmp/mclaren_profiles_with_embeddings.json"
PROFILES_LIGHT = "/tmp/mclaren_profiles.json"

# Sort by driver_id for deterministic output
embedded_profiles_sorted = sorted(embedded_profiles, key=lambda x: x["driver_id"])

# Full file (with embeddings)
with open(PROFILES_FULL, "w", encoding="utf-8") as f:
    json.dump(embedded_profiles_sorted, f, ensure_ascii=False)

# Light file (without embeddings, for human review)
light = [{k: v for k, v in p.items() if k != "embedding" and k != "embedding_error"}
         for p in embedded_profiles_sorted]
with open(PROFILES_LIGHT, "w", encoding="utf-8") as f:
    json.dump(light, f, indent=2, ensure_ascii=False)

print("Local files written.")

# Upload to GCS
bucket = gcs_client.bucket(GCS_BUCKET)
profile_gcs_paths = []

for local, remote_name in [
    (PROFILES_FULL,  "mclaren_profiles_with_embeddings.json"),
    (PROFILES_LIGHT, "mclaren_profiles.json"),
]:
    blob_name = f"{GCS_PROF_PREFIX}/{remote_name}"
    blob = bucket.blob(blob_name)
    blob.upload_from_filename(local, content_type="application/json")
    gcs_uri = f"gs://{GCS_BUCKET}/{blob_name}"
    profile_gcs_paths.append(gcs_uri)
    size_mb = Path(local).stat().st_size / 1024 / 1024
    print(f"  ✅ {remote_name} ({size_mb:.1f} MB) → {gcs_uri}")

print(f"\n✅ Profiles staged to GCS")

---
## Section 5: Verification & Data Manifest

Final checks to confirm everything is in place before lab development begins.

In [ ]:
# Verify GCS Parquet staging: download each file, read with pyarrow,
# and confirm row counts match what we serialised above.

import io
import pyarrow.parquet as pq

print(f"GCS Parquet staging: gs://{GCS_BUCKET}/{GCS_BQ_STAGING}/")
print("=" * 75)
print(f"  {'Table':<30} {'Local rows':>11}  {'GCS rows':>10}  {'Size (KB)':>10}  Status")
print(f"  {'-'*73}")

bucket = gcs_client.bucket(GCS_BUCKET)
all_ok = True

for table_name in sorted(stage_manifest.keys()):
    local_rows = len(stage_manifest[table_name])
    blob_name  = f"{GCS_BQ_STAGING}/{table_name}.parquet"
    blob       = bucket.blob(blob_name)

    try:
        raw = blob.download_as_bytes()
        pq_tbl = pq.read_table(io.BytesIO(raw))
        gcs_rows = pq_tbl.num_rows
        size_kb  = len(raw) / 1024
        match    = gcs_rows == local_rows
        status   = "✅" if match else "❌ MISMATCH"
        if not match:
            all_ok = False
        print(f"  {table_name:<30} {local_rows:>11,}  {gcs_rows:>10,}  {size_kb:>10.0f}  {status}")
    except Exception as e:
        print(f"  {table_name:<30} {local_rows:>11,}  {'—':>10}  {'—':>10}  ❌ {e}")
        all_ok = False

print()
if all_ok:
    print("✅ All Parquet files verified — row counts match")
else:
    print("❌ Some files failed verification — re-run Section 2 to re-stage")


In [ ]:
# Verify GCS: list all staged files and their sizes.

print(f"GCS bucket: gs://{GCS_BUCKET}/{GCS_PREFIX}/")
print("=" * 70)

blobs = list(gcs_client.list_blobs(GCS_BUCKET, prefix=GCS_PREFIX + "/"))
total_mb = 0
for blob in sorted(blobs, key=lambda b: b.name):
    size_mb = blob.size / 1024 / 1024
    total_mb += size_mb
    print(f"  gs://{GCS_BUCKET}/{blob.name:<60} {size_mb:>7.2f} MB")

print(f"\nTotal: {len(blobs)} files, {total_mb:.1f} MB")

In [ ]:
# Print the complete data manifest — a reference card for lab development.
# This summarises every asset, where it lives, and what the lab uses it for.

staging_table_names = sorted(stage_manifest.keys())
fia_names  = [p.replace(f"gs://{GCS_BUCKET}/", "") for p in fia_gcs_paths]
prof_names = [p.replace(f"gs://{GCS_BUCKET}/", "") for p in profile_gcs_paths]

manifest = f"""
╔══════════════════════════════════════════════════════════════════════╗
║          McLaren Race Intelligence Lab — Data Manifest               ║
╠══════════════════════════════════════════════════════════════════════╣
║  Generated : {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M UTC')}                                   ║
║  Project   : {PROJECT_ID:<55}║
╠══════════════════════════════════════════════════════════════════════╣
║  GCS Parquet (BQ staging): gs://{GCS_BUCKET}/{GCS_BQ_STAGING}/      ║
║  (load with: bq load --source_format=PARQUET ...)                   ║
╠══════════════════════════════════════════════════════════════════════╣
"""
for t in staging_table_names:
    manifest += f"║    {t+'.parquet':<66}║\n"

manifest += f"""
╠══════════════════════════════════════════════════════════════════════╣
║  GCS FIA Documents: gs://{GCS_BUCKET}/{GCS_FIA_PREFIX}/             ║
╠══════════════════════════════════════════════════════════════════════╣
"""
for p in fia_names:
    manifest += f"║    {p:<66}║\n"

manifest += f"""
╠══════════════════════════════════════════════════════════════════════╣
║  GCS Profiles: gs://{GCS_BUCKET}/{GCS_PROF_PREFIX}/                 ║
╠══════════════════════════════════════════════════════════════════════╣
"""
for p in prof_names:
    manifest += f"║    {p:<66}║\n"

manifest += f"""
╠══════════════════════════════════════════════════════════════════════╣
║  Driver profiles   : {len(embed_ok):<48}║
║  Embedding dims    : {EMBED_DIM:<48}║
║  McLaren drivers   : {len(mclaren_drivers):<48}║
╚══════════════════════════════════════════════════════════════════════╝

DATA ARCHITECTURE REMINDER FOR LAB DEVELOPMENT:
  • Parquet files in bq-staging/ → students run bq load as Step 1 of the lab.
    This is intentional: loading the data is a teaching moment, not busywork.
  • BQML training views (podium prediction) are built by students, not pre-staged.
  • FIA PDFs in fia-docs/ → Gemini Enterprise data store (RAG, unstructured).
  • JSON profiles in profiles/ → Gemini Enterprise structured data store (retrieval).
  • OpenF1 API (openf1.org) covers 2023+ only — consider as an optional
    enhancement for live race telemetry demos, not core data.
"""

print(manifest)

# Save manifest to GCS for reference
bucket = gcs_client.bucket(GCS_BUCKET)
manifest_blob = bucket.blob(f"{GCS_PREFIX}/DATA_MANIFEST.txt")
manifest_blob.upload_from_string(manifest)
print(f"Manifest saved to gs://{GCS_BUCKET}/{GCS_PREFIX}/DATA_MANIFEST.txt")
